# Phase II: RGB-D/UWB association and particle-filter fusion

This notebook executes the core online path: RGB-D candidate generation, UWB range-consistency gating, and relative-pose particle filtering. The included excerpts are for inspecting behavior only.

In [ ]:
# Install dependencies in Colab; locally use requirements.txt.
import sys
if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib opencv-python-headless

In [ ]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

candidates = [Path.cwd(), Path.cwd().parent,
              Path('/content/drive/MyDrive/rgbd-uwb-fusion')]
ROOT = next((p for p in candidates if (p / 'snippets/phase2_fusion').exists()), None)
assert ROOT is not None, 'Set ROOT to the rgbd-uwb-fusion repository directory'
DATA = ROOT / 'snippets/phase2_fusion'
rng = np.random.default_rng(7)
print('Using', ROOT)

## Generate RGB-D candidates

The detector segments green TurtleBot (not turtlebot exactly in this data but keeping the name for consistency and simplicity) structure in HSV, reads median component depth, applies the measured 0.44 m surface-to-centre correction, and scores the surrounding robot-sized crop. These are proposals, not identity decisions.

In [ ]:
CFG = dict(hue_min=35, hue_max=90, saturation_min=110, value_min=40,
           min_area=4, max_side=200, range_offset=0.44,
           fx=610.832763671875, fy=611.7766723632812, cx=640., cy=360.)

def proposal_box(u, v, depth_m):
    x0 = max(0, round(u - CFG['fx'] * .45 / depth_m))
    x1 = min(1280, round(u + CFG['fx'] * .45 / depth_m))
    y0 = max(0, round(v - CFG['fy'] * .55 / depth_m))
    y1 = min(720, round(v + CFG['fy'] * .15 / depth_m))
    return x0, y0, x1, y1

def generate_candidates(image, depth_mm):
    depth = depth_mm.astype(np.float32) * .001
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (CFG['hue_min'], CFG['saturation_min'], CFG['value_min']),
                       (CFG['hue_max'], 255, 255))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    count, labels, stats, centers = cv2.connectedComponentsWithStats(mask)
    found = []
    for component in range(1, count):
        x, y, w, h, area = map(int, stats[component])
        if area < CFG['min_area'] or w > CFG['max_side'] or h > CFG['max_side']:
            continue
        values = depth[labels == component]
        values = values[values > 0]
        if not values.size:
            continue
        z = float(np.median(values)); u, v = map(float, centers[component])
        right = (u - CFG['cx']) * z / CFG['fx']
        surface_range = math.hypot(z, right)
        corrected_range = surface_range + CFG['range_offset']
        box = proposal_box(u, v, z); x0, y0, x1, y1 = box
        crop = image[y0:y1, x0:x1]
        crop_hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        dark_fraction = float(np.mean(crop_hsv[:, :, 2] < 95))
        edges = cv2.Canny(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), 45, 120)
        edge_density = float(np.count_nonzero(edges) / edges.size)
        score = (.55 * min(1., math.log1p(area) / math.log1p(250.))
                 + .30 * min(1., dark_fraction / .35)
                 + .15 * min(1., edge_density / .10))
        found.append(dict(score=score, area=area, u=u, v=v, box=box,
                          range_m=corrected_range,
                          bearing_deg=math.degrees(math.atan2(-right, z))))
    return sorted(found, key=lambda c: (c['score'], c['area']), reverse=True)

frames = pd.read_csv(DATA / 'rgbd_manifest.csv')
generated = []
for row in frames.itertuples(index=False):
    image = cv2.imread(str(DATA / row.rgb_path))
    depth = cv2.imread(str(DATA / row.depth_path), cv2.IMREAD_UNCHANGED)
    candidates_in_frame = generate_candidates(image, depth)
    generated.append(candidates_in_frame)
print('generated counts:', [len(x) for x in generated])
print('archived full-run counts:', frames.full_run_candidate_count.tolist())
assert [len(x) for x in generated] == frames.full_run_candidate_count.tolist()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), constrained_layout=True)
for ax, row, found in zip(axes.flat, frames.iloc[[0,2,4,6,8,10]].itertuples(),
                          [generated[i] for i in [0,2,4,6,8,10]]):
    image = cv2.cvtColor(cv2.imread(str(DATA / row.rgb_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(image)
    for item in found:
        x0, y0, x1, y1 = item['box']
        ax.add_patch(plt.Rectangle((x0,y0), x1-x0, y1-y0, fill=False,
                                   color='lime', linewidth=1.5))
        ax.text(x0, y0, f"{item['range_m']:.2f}m", color='white',
                bbox={'facecolor':'black', 'alpha':.65, 'pad':2})
    ax.set_title(f'{row.episode}: {len(found)} proposal(s)')
    ax.axis('off')
plt.show()

## Apply the identity gate

For each causally tracked visual event, accept only if the RGB-D range agrees with the synchronized UWB range: `|r_RGBD - r_UWB| <= 1.0 m`, with camera/UWB skew at most 20 ms. RGB-D range performs association; only accepted bearing enters the filter. UWB remains the range measurement.

In [ ]:
data = pd.read_csv(DATA / 'fusion_timeseries.csv')
data['camera_accepted'] = (data.camera_uwb_residual_m.abs() <= 1.0) & (data.uwb_sync_ms.abs() <= 20.0)
print(data.camera_accepted.value_counts(dropna=False))

plt.figure(figsize=(11, 3))
plt.plot(data.time_sec, data.camera_uwb_residual_m.abs(), '.', label='|RGB-D - UWB|')
plt.axhline(1.0, color='crimson', linestyle='--', label='acceptance threshold')
plt.xlabel('bag time (s)'); plt.ylabel('range residual (m)'); plt.legend(); plt.grid(alpha=.25)
plt.show()

## Run a compact particle filter

Particles represent TB2 in the TB1 frame. Both odometry increments drive prediction. UWB updates radial range; accepted camera events update bearing. Ground truth columns are deliberately passed only to the final evaluation cell.

In [ ]:
def wrap(a): return (a + np.pi) % (2*np.pi) - np.pi

def relative_increment(previous, current):
    dx, dy = current[0]-previous[0], current[1]-previous[1]
    c, s = np.cos(previous[2]), np.sin(previous[2])
    return np.array([c*dx+s*dy, -s*dx+c*dy, wrap(current[2]-previous[2])])

def compose_batch(x, delta):
    c, s = np.cos(x[:,2]), np.sin(x[:,2])
    return np.column_stack((x[:,0]+c*delta[0]-s*delta[1],
                            x[:,1]+s*delta[0]+c*delta[1], wrap(x[:,2]+delta[2])))

def left_compose(delta, x):
    c, s = np.cos(delta[2]), np.sin(delta[2])
    return np.column_stack((delta[0]+c*x[:,0]-s*x[:,1],
                            delta[1]+s*x[:,0]+c*x[:,1], wrap(delta[2]+x[:,2])))

def inverse(p):
    c, s = np.cos(p[2]), np.sin(p[2])
    return np.array([-c*p[0]-s*p[1], s*p[0]-c*p[1], -p[2]])

def normalize(w):
    total = w.sum()
    return np.full_like(w, 1/len(w)) if total <= 1e-300 else w/total

def resample(x, w):
    positions = (rng.random() + np.arange(len(w))) / len(w)
    indexes = np.searchsorted(np.cumsum(w), positions)
    return x[indexes].copy(), np.full_like(w, 1/len(w))

def estimate(x, w):
    return np.array([np.sum(w*x[:,0]), np.sum(w*x[:,1]),
                     math.atan2(np.sum(w*np.sin(x[:,2])), np.sum(w*np.cos(x[:,2])))])

def run_filter(use_camera, particles=2500):
    initial = json.loads((DATA / 'initial_state.json').read_text())
    x = np.column_stack((rng.normal(initial['x_m'], .35, particles),
                         rng.normal(initial['y_m'], .35, particles),
                         wrap(rng.normal(initial['yaw_rad'], np.deg2rad(10), particles))))
    w = np.full(particles, 1/particles); output = []
    previous = data.iloc[0]
    for row in data.itertuples(index=False):
        a0 = np.array([previous.tb1_odom_x_m, previous.tb1_odom_y_m, previous.tb1_odom_yaw_rad])
        b0 = np.array([previous.tb2_odom_x_m, previous.tb2_odom_y_m, previous.tb2_odom_yaw_rad])
        a1 = np.array([row.tb1_odom_x_m, row.tb1_odom_y_m, row.tb1_odom_yaw_rad])
        b1 = np.array([row.tb2_odom_x_m, row.tb2_odom_y_m, row.tb2_odom_yaw_rad])
        da, db = relative_increment(a0, a1), relative_increment(b0, b1)
        x = left_compose(inverse(da), compose_batch(x, db))
        x[:,:2] += rng.normal(0, .012, (particles,2)); x[:,2] = wrap(x[:,2] + rng.normal(0, .008, particles))
        range_error = row.uwb_range_m - np.hypot(x[:,0], x[:,1])
        w *= np.maximum(.002, np.exp(-.5*(range_error/.25)**2)); w = normalize(w)
        if use_camera and row.camera_accepted:
            bearing_error = wrap(np.arctan2(x[:,1], x[:,0]) - np.deg2rad(row.camera_bearing_deg))
            w *= np.maximum(.002, np.exp(-.5*(bearing_error/np.deg2rad(8))**2)); w = normalize(w)
        if 1/np.sum(w*w) < particles/2: x, w = resample(x, w)
        output.append(estimate(x,w)); previous = row
    return np.asarray(output)

# Resetting the seed makes the baseline/main comparison deterministic.
rng = np.random.default_rng(7); baseline = run_filter(False)
rng = np.random.default_rng(7); fused = run_filter(True)

In [ ]:
truth = data[['gt_x_m','gt_y_m','gt_yaw_rad']].to_numpy()
def rmse(estimate): return np.sqrt(np.mean(np.sum((estimate[:,:2]-truth[:,:2])**2, axis=1)))
print(f'10 s snippet baseline RMSE: {rmse(baseline):.3f} m')
print(f'10 s snippet fused RMSE:    {rmse(fused):.3f} m')
print('full 110 s baseline/main RMSE: 1.590 m / 1.553 m')
print('full target-visible baseline/main RMSE: 0.560 m / 0.214 m')

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
ax0.plot(truth[:,0], truth[:,1], 'k-', label='protected GT')
ax0.plot(baseline[:,0], baseline[:,1], label='UWB + odometry')
ax0.plot(fused[:,0], fused[:,1], label='+ accepted camera bearing')
ax0.set_aspect('equal'); ax0.set_xlabel('x (m)'); ax0.set_ylabel('y (m)'); ax0.legend(); ax0.grid(alpha=.25)
ax1.plot(data.time_sec, np.linalg.norm(baseline[:,:2]-truth[:,:2], axis=1), label='baseline')
ax1.plot(data.time_sec, np.linalg.norm(fused[:,:2]-truth[:,:2], axis=1), label='fused')
ax1.set_xlabel('bag time (s)'); ax1.set_ylabel('position error (m)'); ax1.legend(); ax1.grid(alpha=.25)
plt.show()

## Use another dataset

Populate `fusion_timeseries.csv` with nanosecond timestamps, planar odometry poses for both robots, synchronized UWB range, and optional camera bearing/range. Ground-truth columns may be omitted while filtering and added later for evaluation. For another camera, change `fx`, `fy`, `cx`, `cy`, HSV limits, and the robot centre offset. Preserve the causal order: propose, track, range-gate, then filter.